# 📈 Efficient Frontier & Mean-Variance Portfolio Optimizer

**A production-quality Colab implementation of Modern Portfolio Theory (Markowitz, 1952)**

---

## 1. Project Overview

This project builds a complete **mean-variance portfolio optimization engine** from historical
market data. Given a basket of assets, it:

- Downloads and cleans historical price data
- Estimates expected returns and the covariance (risk) structure
- Simulates thousands of random portfolios via Monte Carlo
- Solves for the **Maximum Sharpe Ratio** portfolio and the **Global Minimum Variance** portfolio
  using constrained convex optimization
- Plots the full **efficient frontier** with the Capital Allocation Line
- Reports risk/return/allocation metrics in a clean, presentable format

The notebook is self-contained, error-handled, and designed to run top-to-bottom in Google Colab
with zero manual setup beyond installing two packages.

## 2. Real-World Finance Use Case

Every institutional asset allocator — from a two-person RIA to a $500B pension fund's strategic
asset allocation (SAA) committee — starts asset allocation decisions with some version of this
exact math. Concretely, this notebook mirrors:

- **Robo-advisors** (Betterment, Wealthfront) computing model portfolios for each risk tolerance band
- **Investment committees** presenting an efficient frontier chart to justify a target allocation
- **Multi-asset PMs** re-optimizing weights after a quarterly capital markets assumptions update
- **Risk teams** sanity-checking that a proposed portfolio isn't sitting far below the frontier
  (i.e., taking risk with no compensating expected return)

It is intentionally the *baseline* — nearly every more sophisticated allocation technique (risk
parity, Black-Litterman, factor-based allocation) is explicitly built as a response to a
limitation of this exact model.

## 3. System Architecture

```
                    ┌────────────────────┐
                    │   Ticker Universe   │
                    │  (user-defined list)│
                    └──────────┬─────────┘
                               │
                               ▼
                 ┌─────────────────────────┐
                 │   DATA COLLECTION LAYER  │
                 │  yfinance batch download │
                 │   + retry/error handling │
                 └──────────┬───────────────┘
                               │  raw adjusted close prices
                               ▼
                 ┌─────────────────────────┐
                 │  CLEANING & FEATURE      │
                 │  ENGINEERING LAYER       │
                 │  - missing data handling │
                 │  - log returns           │
                 │  - annualized mean/cov   │
                 └──────────┬───────────────┘
                               │  returns matrix, μ, Σ
                               ▼
                 ┌─────────────────────────┐
                 │     CORE MODEL LAYER      │
                 │  - Monte Carlo simulation │
                 │  - SLSQP optimization:    │
                 │    max Sharpe / min var   │
                 │  - efficient frontier     │
                 │    boundary solve         │
                 └──────────┬───────────────┘
                               │  optimal weight vectors
                               ▼
                 ┌─────────────────────────┐
                 │  VISUALIZATION &          │
                 │  REPORTING LAYER          │
                 │  - frontier plot          │
                 │  - allocation charts      │
                 │  - performance metrics    │
                 └────────────────────────────┘
```

**Design principle:** each layer is a pure function that takes the previous layer's output as
input — no global mutable state, no hidden side effects. This means any layer (e.g., the data
source) can be swapped out — say, EDGAR fundamentals instead of yfinance prices — without
touching the optimization or visualization code.

## 4. Required APIs and Data Sources

| Source | Purpose | Auth Required | Notes |
|---|---|---|---|
| **Yahoo Finance** (via `yfinance`) | Daily adjusted close prices | No | Free, unofficial API — can be rate-limited on very large batch pulls; this notebook batches and retries |

This project intentionally uses only Yahoo Finance to keep the notebook runnable with **zero API
keys**. If you want to extend it, natural upgrades include:

- **FRED** — risk-free rate (3-month T-bill) for a more accurate Sharpe ratio, instead of a hardcoded constant
- **Financial Modeling Prep / Alpha Vantage** — fundamentals-based expected return estimates instead of pure historical-mean estimates
- **SEC EDGAR** — company fundamentals if you want to filter the universe by quality factors before optimizing

## 5. Required Python Libraries

| Library | Role |
|---|---|
| `yfinance` | Historical price data download |
| `numpy` | Numerical arrays, linear algebra |
| `pandas` | Data wrangling, time series handling |
| `scipy.optimize` | Constrained optimization (SLSQP) for max Sharpe / min variance |
| `matplotlib` | Charting (frontier, allocation, metrics) |
| `seaborn` | Correlation heatmap styling |

All are pre-installed in Colab except `yfinance`, which the first code cell installs.

## 6. Folder/File Structure

Even though this runs as a single Colab notebook, structuring it as if it were a repo makes it
GitHub-ready and easy to modularize later:

```
efficient-frontier-optimizer/
│
├── README.md                     # Project description, setup, usage
├── efficient_frontier.ipynb      # This notebook (documentation + code)
├── requirements.txt              # yfinance, numpy, pandas, scipy, matplotlib, seaborn
│
├── data/
│   └── prices_cache.csv          # (optional) cached price pull, written by the notebook
│
├── src/                           # If refactored out of the notebook into modules
│   ├── data_pipeline.py          # download_prices(), clean_prices()
│   ├── portfolio_math.py         # compute_returns(), compute_cov(), portfolio_stats()
│   ├── optimizer.py              # max_sharpe(), min_variance(), efficient_frontier()
│   └── visualize.py              # plot_frontier(), plot_allocation()
│
└── outputs/
    ├── efficient_frontier.png
    ├── allocation_pie.png
    └── performance_summary.csv
```

The notebook below keeps everything in one place for Colab convenience, but every function is
written so it could be lifted directly into the `src/` files above with no changes.

## 7. Step-by-Step Build Guide

1. **Install & import** dependencies (Cell A)
2. **Define the universe** — tickers, lookback window, risk-free rate (Cell B)
3. **Data collection pipeline** — download adjusted close prices with retry logic (Cell C)
4. **Data cleaning & feature engineering** — handle missing data, compute log returns, annualize
   mean and covariance (Cell D)
5. **Core models** —
   - Monte Carlo simulation of random portfolios (Cell E)
   - Numerical optimization for Max Sharpe and Min Variance portfolios (Cell F)
   - Efficient frontier boundary via target-return optimization sweep (Cell G)
6. **Visualization** — frontier scatter + boundary curve, allocation pie charts, correlation
   heatmap (Cell H)
7. **Performance metrics** — consolidated summary table for both optimal portfolios (Cell I)
8. **Final deliverables** — exportable CSV/PNG outputs (Cell J)

Each step below is implemented as its own Colab cell with a clear header comment.

## 8–10. Data Collection, Cleaning/Feature Engineering, and Core Models

Implemented directly in code below (Cells A–G). Each cell is commented to explain *why*, not
just *what*.

In [ ]:
# ============================================================
# CELL A — SETUP: Install & Import Dependencies
# ============================================================
!pip install -q yfinance

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.optimize import minimize

# Reproducibility for the Monte Carlo simulation
np.random.seed(42)

# Plot styling — professional, presentation-ready
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
})

print("Environment ready.")

In [ ]:
# ============================================================
# CELL B — CONFIGURATION: Universe, Lookback, Risk-Free Rate
# ============================================================
# Edit this list to build your own universe. A mix of asset classes
# (equities, bonds, gold, real estate) produces a more interesting
# frontier than an all-equity basket.

TICKERS = ["SPY", "QQQ", "TLT", "GLD", "VNQ", "EFA", "IEF", "HYG"]

LOOKBACK_YEARS = 8
RISK_FREE_RATE = 0.045          # annualized; swap for a live FRED 3M T-bill pull if desired
TRADING_DAYS_PER_YEAR = 252
N_MONTE_CARLO_PORTFOLIOS = 15000

END_DATE = pd.Timestamp.today().normalize()
START_DATE = END_DATE - pd.DateOffset(years=LOOKBACK_YEARS)

print(f"Universe: {TICKERS}")
print(f"Window:   {START_DATE.date()} to {END_DATE.date()}")

In [ ]:
# ============================================================
# CELL C — DATA COLLECTION PIPELINE
# ============================================================
def download_prices(tickers, start, end, max_retries=3):
    """
    Download adjusted close prices for a list of tickers with basic
    retry logic, since yfinance batch calls occasionally drop tickers
    on the first attempt.

    Returns
    -------
    pd.DataFrame: dates x tickers, adjusted close prices.
    """
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            raw = yf.download(
                tickers,
                start=start,
                end=end,
                auto_adjust=True,   # adjusted close folded into 'Close'
                progress=False,
                group_by="ticker",
            )
            if raw.empty:
                raise ValueError("Empty dataframe returned by yfinance.")

            # Normalize the multi-index column structure into a flat
            # DataFrame of just closing prices, one column per ticker.
            if isinstance(raw.columns, pd.MultiIndex):
                prices = pd.DataFrame({t: raw[t]["Close"] for t in tickers if t in raw.columns.get_level_values(0)})
            else:
                # Single-ticker edge case
                prices = raw[["Close"]].rename(columns={"Close": tickers[0]})

            missing = set(tickers) - set(prices.columns)
            if missing:
                print(f"Warning: no data returned for {missing}")

            return prices.sort_index()

        except Exception as e:
            last_error = e
            print(f"Attempt {attempt}/{max_retries} failed: {e}")

    raise RuntimeError(f"Failed to download price data after {max_retries} attempts: {last_error}")


prices = download_prices(TICKERS, START_DATE, END_DATE)
print(f"Downloaded {prices.shape[0]} trading days x {prices.shape[1]} tickers")
prices.tail()

In [ ]:
# ============================================================
# CELL D — DATA CLEANING & FEATURE ENGINEERING
# ============================================================
def clean_prices(df, max_missing_frac=0.05):
    """
    Drop tickers with too much missing data, forward-fill small gaps
    (e.g., a single stale trading holiday mismatch), and drop any
    remaining rows with NaNs.
    """
    missing_frac = df.isna().mean()
    bad_tickers = missing_frac[missing_frac > max_missing_frac].index.tolist()
    if bad_tickers:
        print(f"Dropping tickers with >{max_missing_frac:.0%} missing data: {bad_tickers}")
        df = df.drop(columns=bad_tickers)

    df = df.ffill().dropna()
    return df


def compute_returns_and_risk(df, trading_days=TRADING_DAYS_PER_YEAR):
    """
    Compute daily log returns, then annualize expected return and
    the covariance matrix.
    """
    log_returns = np.log(df / df.shift(1)).dropna()

    mean_annual_returns = log_returns.mean() * trading_days
    cov_annual = log_returns.cov() * trading_days

    return log_returns, mean_annual_returns, cov_annual


prices_clean = clean_prices(prices)
log_returns, mu, cov_matrix = compute_returns_and_risk(prices_clean)

assets = list(prices_clean.columns)
n_assets = len(assets)

print("Annualized expected returns:")
print(mu.round(4))
print(f"\nCovariance matrix shape: {cov_matrix.shape}")

In [ ]:
# ============================================================
# CELL E — CORE MODEL: Portfolio Statistics & Monte Carlo Simulation
# ============================================================
def portfolio_performance(weights, mean_returns, cov):
    """Return (expected annual return, annual volatility, Sharpe ratio) for a weight vector."""
    weights = np.array(weights)
    port_return = np.dot(weights, mean_returns)
    port_vol = np.sqrt(weights.T @ cov @ weights)
    sharpe = (port_return - RISK_FREE_RATE) / port_vol if port_vol > 0 else np.nan
    return port_return, port_vol, sharpe


def monte_carlo_portfolios(mean_returns, cov, n_portfolios=N_MONTE_CARLO_PORTFOLIOS):
    """
    Randomly sample long-only, fully-invested portfolios (Dirichlet
    distribution keeps weights valid: non-negative and summing to 1)
    and record their risk/return/Sharpe.
    """
    n = len(mean_returns)
    results = np.zeros((n_portfolios, 3))
    weight_records = np.zeros((n_portfolios, n))

    for i in range(n_portfolios):
        w = np.random.dirichlet(np.ones(n))
        ret, vol, sharpe = portfolio_performance(w, mean_returns, cov)
        results[i] = [ret, vol, sharpe]
        weight_records[i] = w

    return pd.DataFrame(results, columns=["Return", "Volatility", "Sharpe"]), weight_records


try:
    mc_results, mc_weights = monte_carlo_portfolios(mu, cov_matrix)
    print(f"Simulated {len(mc_results):,} random portfolios.")
    mc_results.describe()
except Exception as e:
    raise RuntimeError(f"Monte Carlo simulation failed: {e}")

In [ ]:
# ============================================================
# CELL F — CORE MODEL: Constrained Optimization
# (Max Sharpe Ratio and Global Minimum Variance portfolios)
# ============================================================
def negative_sharpe(weights, mean_returns, cov):
    _, _, sharpe = portfolio_performance(weights, mean_returns, cov)
    return -sharpe


def portfolio_volatility(weights, mean_returns, cov):
    _, vol, _ = portfolio_performance(weights, mean_returns, cov)
    return vol


def solve_optimal_portfolio(objective_fn, mean_returns, cov, extra_constraints=None):
    """
    Generic SLSQP solver: long-only (0 <= w <= 1), fully invested (sum w = 1).
    `extra_constraints` lets us bolt on a target-return constraint for
    frontier sweeps.
    """
    n = len(mean_returns)
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
    if extra_constraints:
        constraints += extra_constraints

    bounds = tuple((0, 1) for _ in range(n))
    init_guess = np.repeat(1 / n, n)

    result = minimize(
        objective_fn,
        init_guess,
        args=(mean_returns, cov),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 500, "ftol": 1e-10},
    )

    if not result.success:
        raise RuntimeError(f"Optimization failed to converge: {result.message}")

    return result.x


try:
    max_sharpe_weights = solve_optimal_portfolio(negative_sharpe, mu, cov_matrix)
    min_vol_weights = solve_optimal_portfolio(portfolio_volatility, mu, cov_matrix)

    ms_ret, ms_vol, ms_sharpe = portfolio_performance(max_sharpe_weights, mu, cov_matrix)
    mv_ret, mv_vol, mv_sharpe = portfolio_performance(min_vol_weights, mu, cov_matrix)

    print("Max Sharpe Portfolio  -> Return: {:.2%}  Vol: {:.2%}  Sharpe: {:.2f}".format(ms_ret, ms_vol, ms_sharpe))
    print("Min Variance Portfolio -> Return: {:.2%}  Vol: {:.2%}  Sharpe: {:.2f}".format(mv_ret, mv_vol, mv_sharpe))
except Exception as e:
    raise RuntimeError(f"Portfolio optimization failed: {e}")

In [ ]:
# ============================================================
# CELL G — CORE MODEL: Efficient Frontier Boundary
# ============================================================
def efficient_frontier(mean_returns, cov, n_points=60):
    """
    Sweep target returns between the min-vol and max-return achievable
    portfolios, minimizing volatility subject to each target return.
    This traces the *true* analytical frontier boundary, which the
    Monte Carlo cloud can only approximate.
    """
    target_returns = np.linspace(mean_returns.min(), mean_returns.max(), n_points)
    frontier_vols = []

    for target in target_returns:
        constraint = [{"type": "eq", "fun": lambda w, t=target: np.dot(w, mean_returns) - t}]
        try:
            w = solve_optimal_portfolio(portfolio_volatility, mean_returns, cov, extra_constraints=constraint)
            _, vol, _ = portfolio_performance(w, mean_returns, cov)
            frontier_vols.append(vol)
        except RuntimeError:
            frontier_vols.append(np.nan)

    frontier_df = pd.DataFrame({"Return": target_returns, "Volatility": frontier_vols}).dropna()
    return frontier_df


frontier_df = efficient_frontier(mu, cov_matrix)
print(f"Solved {len(frontier_df)} points on the efficient frontier boundary.")

## 11–12. Visualizations, Dashboard Components & Performance Metrics

The next cells produce three presentation-ready charts and a consolidated performance table.

In [ ]:
# ============================================================
# CELL H — VISUALIZATION: Efficient Frontier
# ============================================================
fig, ax = plt.subplots(figsize=(11, 7))

sc = ax.scatter(
    mc_results["Volatility"], mc_results["Return"],
    c=mc_results["Sharpe"], cmap="viridis", s=8, alpha=0.35, label="Simulated portfolios"
)
cbar = fig.colorbar(sc)
cbar.set_label("Sharpe Ratio")

ax.plot(frontier_df["Volatility"], frontier_df["Return"], color="black", linewidth=2.2,
        label="Efficient Frontier (analytical)")

ax.scatter(ms_vol, ms_ret, color="crimson", marker="*", s=450, edgecolor="black",
           linewidth=0.8, label="Max Sharpe Portfolio", zorder=5)
ax.scatter(mv_vol, mv_ret, color="dodgerblue", marker="D", s=140, edgecolor="black",
           linewidth=0.8, label="Min Variance Portfolio", zorder=5)

# Capital Allocation Line through the risk-free rate and max Sharpe portfolio
cal_x = np.linspace(0, mc_results["Volatility"].max(), 50)
cal_y = RISK_FREE_RATE + ms_sharpe * cal_x
ax.plot(cal_x, cal_y, linestyle="--", color="gray", linewidth=1.4, label="Capital Allocation Line")

ax.set_xlabel("Annualized Volatility (Risk)")
ax.set_ylabel("Annualized Expected Return")
ax.set_title("Efficient Frontier — Mean-Variance Optimization")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(loc="lower right", frameon=True)

plt.tight_layout()
plt.savefig("efficient_frontier.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL I — VISUALIZATION: Allocation Breakdown & Correlation Heatmap
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

def plot_allocation(ax, weights, title):
    w = pd.Series(weights, index=assets)
    w = w[w > 0.005]  # hide near-zero slivers for readability
    ax.pie(w, labels=w.index, autopct="%1.1f%%", startangle=90,
           wedgeprops={"edgecolor": "white", "linewidth": 1})
    ax.set_title(title)

plot_allocation(axes[0], max_sharpe_weights, "Max Sharpe Allocation")
plot_allocation(axes[1], min_vol_weights, "Min Variance Allocation")

corr = log_returns.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[2],
            cbar_kws={"label": "Correlation"})
axes[2].set_title("Asset Correlation Matrix")

plt.tight_layout()
plt.savefig("allocation_and_correlation.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL J — PERFORMANCE METRICS & FINAL DELIVERABLES EXPORT
# ============================================================
def summarize_portfolio(weights, mean_returns, cov, label):
    ret, vol, sharpe = portfolio_performance(weights, mean_returns, cov)
    row = {"Portfolio": label, "Expected Return": ret, "Volatility": vol, "Sharpe Ratio": sharpe}
    row.update({f"Weight: {a}": w for a, w in zip(assets, weights)})
    return row

summary = pd.DataFrame([
    summarize_portfolio(max_sharpe_weights, mu, cov_matrix, "Max Sharpe"),
    summarize_portfolio(min_vol_weights, mu, cov_matrix, "Min Variance"),
])

display_cols = ["Portfolio", "Expected Return", "Volatility", "Sharpe Ratio"]
summary_display = summary[display_cols].copy()
summary_display["Expected Return"] = summary_display["Expected Return"].map("{:.2%}".format)
summary_display["Volatility"] = summary_display["Volatility"].map("{:.2%}".format)
summary_display["Sharpe Ratio"] = summary_display["Sharpe Ratio"].map("{:.2f}".format)

print("=" * 60)
print("PORTFOLIO PERFORMANCE SUMMARY")
print("=" * 60)
print(summary_display.to_string(index=False))

# Export deliverables
summary.to_csv("performance_summary.csv", index=False)
prices_clean.to_csv("prices_cache.csv")

print("\nSaved: efficient_frontier.png, allocation_and_correlation.png, performance_summary.csv, prices_cache.csv")

## 13. Final Deliverables

Running this notebook end-to-end produces:

- `efficient_frontier.png` — the frontier chart with Monte Carlo cloud, analytical boundary, Max Sharpe/Min Variance markers, and CAL
- `allocation_and_correlation.png` — allocation pie charts for both optimal portfolios plus a correlation heatmap
- `performance_summary.csv` — return, volatility, Sharpe ratio, and full weight vectors for both portfolios
- `prices_cache.csv` — the cleaned price history used, for reproducibility
- A fully documented, GitHub-ready notebook with clear section headers, error handling on every
  external call (data download, optimization convergence), and no hardcoded magic numbers outside
  the configuration cell

## 14. Resume Description

> **Mean-Variance Portfolio Optimizer (Python, NumPy, SciPy)**
> Built an end-to-end portfolio optimization engine implementing Modern Portfolio Theory: pulled
> and cleaned multi-asset historical price data, estimated annualized return/covariance
> statistics, ran a 15,000-portfolio Monte Carlo simulation, and solved constrained SLSQP
> optimizations for the Maximum Sharpe Ratio and Global Minimum Variance portfolios. Produced the
> analytical efficient frontier boundary and a presentation-ready visualization suite (frontier
> plot, allocation breakdown, correlation heatmap) suitable for an investment committee readout.

## 15. Potential Upgrades

- **Black-Litterman blending** — replace pure historical-mean return estimates with a
  market-equilibrium prior blended with subjective views, which fixes MVO's well-known sensitivity
  to noisy expected-return inputs
- **Shrinkage covariance estimation** (Ledoit-Wolf) — more stable covariance estimates than the
  raw sample covariance, especially with a short lookback or many assets
- **Live risk-free rate** — pull the 3-month T-bill yield from FRED instead of hardcoding it
- **Transaction cost / turnover constraints** — penalize large rebalances from a current holdings
  vector, closer to how a live portfolio would actually be optimized
- **Rolling re-optimization backtest** — re-run the optimizer on a rolling window to see how
  realized (not just in-sample) performance compares to a static 60/40 benchmark
- **Streamlit/Dash wrapper** — turn this into an interactive tool where a user can drag risk
  tolerance and see the resulting allocation update live
- **Factor-based expected returns** — feed in Fama-French-style factor forecasts instead of pure
  historical means for the expected-return vector